# Overfitting, underfitting & learning curves

Learning curves diagnose whether error is dominated by bias, variance, or data scarcity.

Learning curves compare training error to validation error as the amount of data grows. A wide gap suggests variance; high errors on both sides suggest bias or missing signal. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_wine
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def clf_ladder():
    """D1..D5 classification ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [0.4, 0.2], [3.0, 3.0], [2.6, 3.2]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 hand 2-D points", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=0.8, random_state=1)
    rungs.append(("D2 clean blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.28, random_state=2)
    rungs.append(("D3 noisy moons (non-linear)", x3, y3))

    wine = load_wine()
    rungs.append(("D4 Wine (real, 13-D, 3-class)", wine.data, wine.target))

    cancer = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (real, 30-D)", cancer.data, cancer.target))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def logistic_baseline(x_tr, y_tr, x_te):
    model = LogisticRegression(max_iter=2000)
    model.fit(x_tr, y_tr)
    return model.predict(x_te)


def split_scaled(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=3, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_logistic(x_tr, y_tr, C=1.0):
    model = LogisticRegression(C=C, max_iter=2500)
    model.fit(x_tr, y_tr)
    return model


def classification_error(y_true, y_pred):
    return float(1.0 - accuracy_score(y_true, y_pred))


def lesson_score(losses, cost, alternative):
    raw = round(float(np.mean(losses)), 3)
    score = round(raw + cost, 3)
    gap = round(alternative - score, 3)
    relative_gap = gap / alternative
    stabilized = 0.8 * score
    return {
        "raw": raw,
        "cost": cost,
        "score": score,
        "alternative": alternative,
        "gap": gap,
        "relative_gap": relative_gap,
        "stabilized": stabilized,
    }


def preview_ladder(rungs):
    for name, X, y in rungs:
        labels, counts = np.unique(y, return_counts=True)
        info = dict(zip(labels.tolist(), counts.tolist()))
        print(f"{name}: X={X.shape}, classes={info}, sample={np.round(X[:2], 3).tolist()}")


## The concept, built once on D1

The lesson formula is $$gap(m)=R_{val}(m)-R_{train}(m)$$. We first rebuild the exact lesson arithmetic before using a reusable method on larger data.

In [ ]:

def overfitting_underfitting_learning_curves_method(losses, cost, alternative):
    """Recompute the lesson raw average, cost-aware score, and alternative gap."""
    losses = np.asarray(losses, dtype=float)
    summary = lesson_score(losses, cost, alternative)
    return summary


losses = np.array([0.246, 0.109, 0.539], dtype=float)
summary = overfitting_underfitting_learning_curves_method(losses, cost=0.050, alternative=0.392)

assert abs(summary["raw"] - 0.298) < 1e-12
assert abs(summary["score"] - 0.348) < 1e-12
assert abs(summary["gap"] - 0.044) < 1e-12
assert abs(summary["relative_gap"] - 0.112) < 0.001
assert abs(summary["stabilized"] - 0.278) < 0.001

print("losses:", losses.tolist())
print("raw average:", round(summary["raw"], 3))
print("score with cost:", round(summary["score"], 3))
print("gap to alternative:", round(summary["gap"], 3))
print("relative gap:", round(summary["relative_gap"], 3))


The same score object also carries the stabilized launch number and, for this topic, the topic-specific metric calculation used on the toy D1 counts or residuals.

In [ ]:

def learning_curve_gap(train_losses, validation_losses):
    train_losses = np.asarray(train_losses, dtype=float)
    validation_losses = np.asarray(validation_losses, dtype=float)
    return validation_losses - train_losses

toy_train = np.array([0.246, 0.200, 0.180])
toy_validation = np.array([0.539, 0.420, 0.348])
toy_gap = learning_curve_gap(toy_train, toy_validation)
assert round(float(toy_gap[0]), 3) == 0.293


print("toy learning-curve gaps:", np.round(toy_gap, 3).tolist())
print("first gap = 0.539 - 0.246:", round(float(toy_gap[0]), 3))


## The dataset ladder

The notebook embeds the shared `clf_ladder()` source so it is self-contained in Colab. D1 is inspectable; D5 is real, higher-dimensional, and imbalanced enough to make the checks matter.

In [ ]:

rungs = clf_ladder()
preview_ladder(rungs)


## Run learning-curve gaps across D1–D5

For each rung we train on increasing fractions of the training split and track the final validation-minus-training loss gap.

In [ ]:

def learning_curve_rung(X, y):
    x_tr, x_te, y_tr, y_te = split_scaled(X, y)
    fractions = np.array([0.35, 0.55, 0.75, 1.0])
    train_losses = []
    validation_losses = []
    for frac in fractions:
        size = max(len(np.unique(y_tr)) * 2, int(len(y_tr) * frac))
        x_sub = x_tr[:size]
        y_sub = y_tr[:size]
        if len(np.unique(y_sub)) < len(np.unique(y_tr)):
            x_sub = x_tr
            y_sub = y_tr
        model = fit_logistic(x_sub, y_sub, C=1.0)
        train_losses.append(classification_error(y_sub, model.predict(x_sub)))
        validation_losses.append(classification_error(y_te, model.predict(x_te)))
    gaps = learning_curve_gap(train_losses, validation_losses)
    return fractions, np.array(train_losses), np.array(validation_losses), gaps

results = []
for name, X, y in rungs:
    fractions, train_losses, validation_losses, gaps = learning_curve_rung(X, y)
    metric = float(gaps[-1])
    acc = clf_accuracy(logistic_baseline, X, y)
    results.append({"name": name, "fractions": fractions, "train": train_losses, "validation": validation_losses, "gap": metric, "accuracy": acc})
    print(f"{name}: final_gap={metric:.3f}, final_val_loss={validation_losses[-1]:.3f}, clf_accuracy={acc:.3f}")


In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for ax, row in zip(axes[0], results):
    ax.plot(row["fractions"], row["train"], marker="o", label="train")
    ax.plot(row["fractions"], row["validation"], marker="o", label="validation")
    ax.set_title(row["name"].split(" (")[0], fontsize=8)
    ax.set_ylim(-0.02, 1.02)
axes[0, 0].legend(fontsize=8)
axes[1, 0].plot(range(1, 6), [row["gap"] for row in results], marker="o")
axes[1, 0].set_title("final gap vs rung")
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("val - train loss")
for ax in axes[1, 1:]:
    ax.axis("off")
plt.tight_layout()
plt.show()


## Pitfall on D5: optimizing the raw term and forgetting the cost

The smallest training loss is not the safest model. The fix is to compare validation gap and complexity cost before selecting.

In [ ]:

name, X, y = rungs[-1]
x_tr, x_te, y_tr, y_te = split_scaled(X, y)
rows = []
for C in [0.01, 0.1, 1.0, 10.0, 100.0]:
    model = fit_logistic(x_tr, y_tr, C=C)
    preds = model.predict(x_te)
    raw_loss = classification_error(y_te, preds)
    complexity_cost = 0.050 * math.log10(C + 1.0) / math.log10(11.0)
    decision_score = raw_loss + complexity_cost
    rows.append((C, raw_loss, complexity_cost, decision_score))

raw_best = min(rows, key=lambda row: row[1])
fixed_best = min(rows, key=lambda row: row[3])
print("D5 candidate table: C, raw validation loss, cost, decision score")
for row in rows:
    print(tuple(round(value, 4) for value in row))
print("wrong raw-only choice:", raw_best)
print("cost-aware choice:", fixed_best)
print("decision-score improvement:", round(raw_best[3] - fixed_best[3], 4))
assert fixed_best[3] <= raw_best[3] + 1e-12


## Evaluate it + Practice

- Track `validation gap` against a no-skill baseline before trusting the result.
- Sanity-check that D1 reproduces the lesson numbers exactly and that D5 is not a toy shortcut.
- Ablation: choose the model with the lowest training loss; the metric should get worse or the selected model should become less stable.
- Failure signals: large validation gap, unstable threshold/criterion, or a cost-aware score that disagrees with the raw metric.
- Report both the raw metric and the cost-aware decision score when the lesson formula includes a cost.

Practice prompts:
1. Change one candidate hyperparameter and recompute the D5 table.
2. Replace the no-skill baseline and explain whether `validation gap` moved for the right reason.
3. Add one diagnostic plot that would catch the named pitfall for learning curves.